# Stochastic Calculus, Step by Step · Lesson 35: Regression and PDEs, further

Companion notebook for the lesson. Every number in the lesson's text, tables and figures is produced by running this notebook and read from `build/outputs.json`.

> **The course's numbers.** A share priced at S₀ = 100, a strike K = 105, a risk-free rate r = 3% (continuously compounded), a volatility σ = 20% and a horizon T = 1 year. Where a lesson needs the share's real-world expected return, it is μ = 8%. Made up for teaching; every random number is simulated with a fixed seed.
>
> Educational material only, not financial advice.

1. Setup · 2. Regression as conditional expectation · 3. Kernel regression · 4. The forward Kolmogorov equation · 5. Two dimensions: the ADI scheme · 6. Exercises · 7. Export

**Running in Google Colab?** Nothing to install: this notebook only needs numpy, pandas and matplotlib, which Colab already has. Run the cells in order.

In [ ]:
# Parameters (papermill overrides these)
output_json = "build/outputs.json"
seed = 2026

## 1. Setup

In [ ]:
import json
import math
import datetime as dt
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

S0, K, r, sigma, T = 100.0, 105.0, 0.03, 0.20, 1.0     # the course's numbers
N = lambda x: 0.5 * (1 + math.erf(x / math.sqrt(2)))   # standard normal distribution function
out = {"meta": {
    "series": "Stochastic Calculus, Step by Step",
    "lesson": 35,
    "quantlib_version": "not used",
    "quotes_label": "simulated (no market data)",
    "seed": seed,
    "generated_at": dt.datetime.now().isoformat(timespec="seconds"),
}, "inputs": {"S0": S0, "K": K, "r": r, "sigma": sigma, "T": T, "r_pct": r * 100, "sigma_pct": sigma * 100}}

In [ ]:
mu = 0.08                         # the share's real-world expected return
theta_mpr = (mu - r) / sigma      # the market price of risk
Nv = np.vectorize(N)              # N, for arrays
phi = lambda x: np.exp(-np.asarray(x, float) ** 2 / 2) / math.sqrt(2 * math.pi)   # the normal density


def d_plus(s, tau, sig=sigma, k=K):
    return (np.log(s / k) + (r + 0.5 * sig**2) * tau) / (sig * np.sqrt(tau))


def d_minus(s, tau, sig=sigma, k=K):
    return d_plus(s, tau, sig, k) - sig * np.sqrt(tau)


def call(s, tau, sig=sigma, k=K):
    """The Black-Scholes call price: Lesson 20 derives it."""
    return s * Nv(d_plus(s, tau, sig, k)) - k * np.exp(-r * tau) * Nv(d_minus(s, tau, sig, k))


def put(s, tau, sig=sigma, k=K):
    return call(s, tau, sig, k) - s + k * np.exp(-r * tau)


out["bs"] = {"mu": mu, "theta": theta_mpr, "d_plus": float(d_plus(S0, T)), "d_minus": float(d_minus(S0, T)),
             "N_plus": N(float(d_plus(S0, T))), "N_minus": N(float(d_minus(S0, T))),
             "call": float(call(S0, T)), "put": float(put(S0, T))}
out["bs"]

## 2. Regression as conditional expectation

Simulate the share to $t = 0.5$ and on to $T = 1$. The discounted put payoff $Y = e^{-r(T - t)}\max(K - S_T, 0)$, regressed on functions of $S_t$, estimates $E[Y \mid S_t]$. Here we know the exact answer: the Black-Scholes put with half a year left. Fit polynomials of degree 1 to 5, on all paths and on the in-the-money ones only, and measure the error against the exact curve.

In [ ]:
t_mid = 0.5
rng = np.random.default_rng(seed)
M_r = 20_000
S_t = S0 * np.exp((r - 0.5 * sigma**2) * t_mid + sigma * math.sqrt(t_mid) * rng.standard_normal(M_r))
S_T = S_t * np.exp((r - 0.5 * sigma**2) * (T - t_mid) + sigma * math.sqrt(T - t_mid) * rng.standard_normal(M_r))
Y = math.exp(-r * (T - t_mid)) * np.maximum(K - S_T, 0)
exact_cv = put(S_t, T - t_mid)
itm = S_t < K
reg_rows, fits = [], {}
for deg in (1, 2, 3, 4, 5):
    for which, mask in (("all", np.ones(M_r, bool)), ("itm", itm)):
        coef = np.polyfit(S_t[mask] / K, Y[mask], deg)
        fitted = np.polyval(coef, S_t[itm] / K)
        rmse = float(np.sqrt(((fitted - exact_cv[itm]) ** 2).mean()))
        reg_rows.append({"degree": deg, "paths": which, "rmse_itm": rmse})
        fits[(deg, which)] = coef
grid_s = np.linspace(70, 105, 71)
pick = rng.choice(M_r, 1500, replace=False)
wide = [{"degree": d, "all": next(x["rmse_itm"] for x in reg_rows if x["degree"] == d and x["paths"] == "all"),
         "itm": next(x["rmse_itm"] for x in reg_rows if x["degree"] == d and x["paths"] == "itm")} for d in (1, 2, 3, 4, 5)]
out["reg"] = {"paths": M_r, "t": t_mid, "itm_share": float(itm.mean()), "rows": reg_rows, "wide": wide,
              "noise_sd": float((Y - exact_cv).std()),
              "fig": {"S": S_t[pick].round(3).tolist(), "Y": Y[pick].round(4).tolist(), "grid": grid_s.tolist(),
                      "exact": put(grid_s, T - t_mid).round(5).tolist(),
                      "deg1": np.polyval(fits[(1, "itm")], grid_s / K).round(5).tolist(),
                      "deg3": np.polyval(fits[(3, "itm")], grid_s / K).round(5).tolist()}}
pd.DataFrame(reg_rows).pivot(index="degree", columns="paths", values="rmse_itm").round(4)

## 3. Kernel regression

A polynomial fixes the shape in advance. A **kernel regression** (Nadaraya-Watson) estimates $E[Y \mid X = x]$ as a weighted average of the $Y$'s, with weights that fall off with the distance between $X_i$ and $x$: $\hat m(x) = \sum_iK_h(x - X_i)Y_i / \sum_iK_h(x - X_i)$, with a normal-shaped $K_h$ of width $h$, the **bandwidth**. Test it on $E[S_T^2 \mid S_{0.5}] = S_{0.5}^2e^{(2r + \sigma^2)(T - 0.5)}$ (Lesson 17), with 5,000 paths.

In [ ]:
rng = np.random.default_rng(seed + 1)
M_k = 5_000
Xk = S0 * np.exp((r - 0.5 * sigma**2) * t_mid + sigma * math.sqrt(t_mid) * rng.standard_normal(M_k))
Yk = (Xk * np.exp((r - 0.5 * sigma**2) * (T - t_mid) + sigma * math.sqrt(T - t_mid) * rng.standard_normal(M_k))) ** 2
xg = np.linspace(80, 125, 46)
exact_k = xg**2 * math.exp((2 * r + sigma**2) * (T - t_mid))


def nadaraya_watson(x, X, Yv, h):
    w = np.exp(-0.5 * ((x[:, None] - X[None, :]) / h) ** 2)
    return (w * Yv).sum(axis=1) / w.sum(axis=1)


kb_rows, kb_fig = [], {}
for h in (0.5, 2.0, 5.0, 15.0, 40.0):
    est = nadaraya_watson(xg, Xk, Yk, h)
    kb_rows.append({"h": h, "rmse": float(np.sqrt(((est - exact_k) ** 2).mean())),
                    "bias_ends": float(0.5 * ((est[0] - exact_k[0]) + (est[-1] - exact_k[-1])))})
    if h in (0.5, 5.0, 40.0):
        kb_fig[f"h{h:g}".replace(".", "_")] = est.round(2).tolist()
best_h = min(kb_rows, key=lambda d: d["rmse"])["h"]
out["kern"] = {"paths": M_k, "rows": kb_rows, "best_h": best_h, "growth": math.exp((2 * r + sigma**2) * (T - t_mid)),
               "fig": {"x": xg.tolist(), "exact": exact_k.round(2).tolist(), **kb_fig}}
pd.DataFrame(kb_rows).round(2)

## 4. The forward Kolmogorov equation

Feynman-Kac's equation (Lesson 28) runs **backwards** from the payoff. The density $p(t, x)$ of $X_t = \ln S_t$ obeys an equation that runs **forwards** from today, the **Fokker-Planck** equation:

$$p_t = -\nu\,p_x + \tfrac12\sigma^2p_{xx}, \qquad \nu = r - \tfrac12\sigma^2 .$$

Start from the exact density at a small time, step it forward to $T$ on a grid (Crank-Nicolson, Lesson 29, with the tridiagonal solver), and compare with the normal density. Then price the call from the density: one forward solve, then an integral.

In [ ]:
def thomas(a, b, c, d):
    # Tridiagonal solve (Lesson 29); a, b, c are the three diagonals, d may hold many right-hand sides as columns.
    n = len(b); cp = np.zeros(n); dp = np.zeros_like(d, dtype=float)
    cp[0] = c[0] / b[0]; dp[0] = d[0] / b[0]
    for i in range(1, n):
        m = b[i] - a[i] * cp[i - 1]
        cp[i] = c[i] / m if i < n - 1 else 0.0
        dp[i] = (d[i] - a[i] * dp[i - 1]) / m
    x = np.zeros_like(dp)
    x[-1] = dp[-1]
    for i in range(n - 2, -1, -1):
        x[i] = dp[i] - cp[i] * x[i + 1]
    return x


nu = r - 0.5 * sigma**2
t0 = 0.02
Mx, Nt = 600, 400
xg_fp = math.log(S0) + np.linspace(-6 * sigma, 6 * sigma, Mx + 1)
dx = xg_fp[1] - xg_fp[0]
dens = lambda t: np.exp(-(xg_fp - math.log(S0) - nu * t) ** 2 / (2 * sigma**2 * t)) / math.sqrt(2 * math.pi * sigma**2 * t)
p = dens(t0)
dt = (T - t0) / Nt
al = 0.5 * sigma**2 / dx**2; be = nu / (2 * dx)
lo_, mid_, up_ = al + be, -2 * al, al - be      # coefficients of p[i-1], p[i], p[i+1] in the right-hand side
n_ = Mx + 1
a_ = np.full(n_, -0.5 * dt * lo_); b_ = np.full(n_, 1 - 0.5 * dt * mid_); c_ = np.full(n_, -0.5 * dt * up_)
a_[0] = c_[0] = a_[-1] = c_[-1] = 0.0; b_[0] = b_[-1] = 1.0
snaps = {}
for n in range(1, Nt + 1):
    rhs = p.copy()
    rhs[1:-1] = p[1:-1] + 0.5 * dt * (lo_ * p[:-2] + mid_ * p[1:-1] + up_ * p[2:])
    rhs[0] = rhs[-1] = 0.0
    p = thomas(a_, b_, c_, rhs)
    tn = t0 + n * dt
    for mark in (0.25, 1.0):
        if abs(tn - mark) < dt / 2:
            snaps[mark] = p.copy()
w_trap = np.full(n_, dx); w_trap[[0, -1]] /= 2
price_fp = math.exp(-r * T) * float((np.maximum(np.exp(xg_fp) - K, 0) * p * w_trap).sum())
out["fp"] = {"t0": t0, "M": Mx, "N": Nt, "mass": float((p * w_trap).sum()), "mean_S": float((np.exp(xg_fp) * p * w_trap).sum()),
             "forward": S0 * math.exp(r * T), "max_err": float(np.abs(p - dens(T)).max()), "peak": float(dens(T).max()),
             "price": price_fp, "bs": float(call(S0, T)), "error": price_fp - float(call(S0, T)),
             "fig": {"x": np.exp(xg_fp[::4]).round(3).tolist(), "p0": dens(t0)[::4].round(4).tolist(),
                     "p25": snaps[0.25][::4].round(4).tolist(), "p1": p[::4].round(4).tolist(), "e1": dens(T)[::4].round(4).tolist()}}
{k: v for k, v in out["fp"].items() if k != "fig"}

## 5. Two dimensions: the ADI scheme

An option on two shares solves a PDE in two space variables. In logs, $x = \ln S_1$, $y = \ln S_2$:

$$U_\tau = \nu_1U_x + \tfrac12\sigma_1^2U_{xx} + \nu_2U_y + \tfrac12\sigma_2^2U_{yy} + \rho\sigma_1\sigma_2U_{xy} - rU .$$

A fully implicit step would need a large system that isn't tridiagonal. The **ADI** (alternating direction implicit) schemes take an explicit step with everything, then correct it implicitly in $x$ alone and in $y$ alone: two batches of tridiagonal solves. The cross term stays explicit. **Douglas** stops there; **Craig-Sneyd** updates the cross term with the new values and repeats the two sweeps. Test both on Lesson 27's exchange option, $\max(S_1 - S_2, 0)$ with $S_1 = 100$, $S_2 = 95$, $\sigma_1 = 20\%$, $\sigma_2 = 30\%$, $\rho = 0.5$, whose Margrabe price is known. Two details from Lessons 22 and 29 matter again: the grid uses one spacing $h$ in both directions, chosen so that the payoff's kink ($S_1 = S_2$) runs through grid nodes, and the first four steps are fully implicit, to damp the kink's zigzag.

In [ ]:
a1, a2, s1_, s2_, rho12 = 100.0, 95.0, 0.20, 0.30, 0.5
sx2 = s1_**2 + s2_**2 - 2 * rho12 * s1_ * s2_
dd1 = (math.log(a1 / a2) + 0.5 * sx2 * T) / math.sqrt(sx2 * T)
margrabe = a1 * N(dd1) - a2 * N(dd1 - math.sqrt(sx2 * T))


def adi_exchange(m, N_, scheme="craig-sneyd", half=1.5, damp=4):
    # Grid spacing h = ln(a1/a2)/m in both directions, so the kink S1 = S2 runs through nodes; +-1.5 in each log.
    h = math.log(a1 / a2) / m
    n_half = math.ceil(half / h)
    x = math.log(a1) + h * np.arange(-n_half, n_half + 1)
    y = math.log(a2) + h * np.arange(-n_half, n_half + 1)
    M_ = len(x) - 1
    k = T / N_
    nu1, nu2 = r - 0.5 * s1_**2, r - 0.5 * s2_**2
    X_, Y_ = np.meshgrid(x, y, indexing="ij")
    U = np.maximum(np.exp(X_) - np.exp(Y_), 0)
    edge = U.copy()                                  # boundary values: the payoff (far from today's prices)

    def A1(V):                                       # the x terms, and half the discount
        o = np.zeros_like(V)
        o[1:-1] = nu1 * (V[2:] - V[:-2]) / (2 * h) + 0.5 * s1_**2 * (V[2:] - 2 * V[1:-1] + V[:-2]) / h**2 - 0.5 * r * V[1:-1]
        return o

    def A2(V):                                       # the y terms, and the other half
        o = np.zeros_like(V)
        o[:, 1:-1] = (nu2 * (V[:, 2:] - V[:, :-2]) / (2 * h) + 0.5 * s2_**2 * (V[:, 2:] - 2 * V[:, 1:-1] + V[:, :-2]) / h**2
                      - 0.5 * r * V[:, 1:-1])
        return o

    def A0(V):                                       # the cross term
        o = np.zeros_like(V)
        o[1:-1, 1:-1] = rho12 * s1_ * s2_ * (V[2:, 2:] - V[2:, :-2] - V[:-2, 2:] + V[:-2, :-2]) / (4 * h * h)
        return o

    def implicit(nu_, s_, rhs, th):
        # (I - th k A) V = rhs along every line at once, with the edge rows held at the boundary values
        lo, mi, up = 0.5 * s_**2 / h**2 - nu_ / (2 * h), -s_**2 / h**2 - 0.5 * r, 0.5 * s_**2 / h**2 + nu_ / (2 * h)
        n = M_ + 1
        a = np.full(n, -th * k * lo); b = np.full(n, 1 - th * k * mi); c = np.full(n, -th * k * up)
        a[0] = c[0] = a[-1] = c[-1] = 0.0; b[0] = b[-1] = 1.0
        return thomas(a, b, c, rhs)

    def sweeps(Y0, a1U, a2U, th):
        Y1 = implicit(nu1, s1_, Y0 - th * k * a1U, th)               # solve along x
        Y1[0], Y1[-1] = edge[0], edge[-1]
        Y2 = implicit(nu2, s2_, (Y1 - th * k * a2U).T, th).T         # then along y
        Y2[:, 0], Y2[:, -1] = edge[:, 0], edge[:, -1]
        Y2[0], Y2[-1] = edge[0], edge[-1]
        return Y2

    for step in range(N_):
        th = 1.0 if step < damp else 0.5
        a0U, a1U, a2U = A0(U), A1(U), A2(U)
        Y0 = U + k * (a0U + a1U + a2U)
        U_new = sweeps(Y0, a1U, a2U, th)
        if scheme == "craig-sneyd":                                  # refresh the cross term and sweep again
            U_new = sweeps(Y0 + 0.5 * k * (A0(U_new) - a0U), a1U, a2U, th)
        U = U_new
    c_explicit = k * (s1_**2 + s2_**2) / h**2
    return float(U[n_half, n_half]), U, x, y, M_, c_explicit


adi_rows = []
for m in (1, 2, 4):
    N_ = 2 * math.ceil(1.5 / (math.log(a1 / a2) / m))       # as many time steps as grid steps
    vd, *_ = adi_exchange(m, N_, "douglas")
    vc, U_, xx_, yy_, M_, c_ex = adi_exchange(m, N_, "craig-sneyd")
    adi_rows.append({"m": m, "h": math.log(a1 / a2) / m, "M": M_, "N": N_, "points": (M_ + 1) ** 2,
                     "douglas": vd - margrabe, "craig_sneyd": vc - margrabe, "c": c_ex, "explicit_N": math.ceil(N_ * c_ex)})
v_fig, U_fig, x_fig, y_fig, _, _ = adi_exchange(2, 118, "craig-sneyd")
sel = slice(None, None, 4)
out["adi"] = {"margrabe": margrabe, "sx": math.sqrt(sx2), "rows": adi_rows,
              "ratio_d": adi_rows[1]["douglas"] / adi_rows[2]["douglas"], "ratio_cs": adi_rows[1]["craig_sneyd"] / adi_rows[2]["craig_sneyd"],
              "fig": {"s1": np.exp(x_fig[sel]).round(3).tolist(), "s2": np.exp(y_fig[sel]).round(3).tolist(), "U": U_fig[sel, sel].round(3).tolist(),
                      "M": [d["M"] for d in adi_rows], "douglas": [abs(d["douglas"]) for d in adi_rows],
                      "cs": [abs(d["craig_sneyd"]) for d in adi_rows]}}
pd.DataFrame(adi_rows)

In [ ]:
# Chart data for the video (no random numbers)
out["reg"]["fig"]["ticks"] = [{"x": float(v), "label": f"{v}"} for v in (70, 80, 90, 100, 105)]
out["kern"]["fig"]["ticks"] = [{"x": float(v), "label": f"{v}"} for v in (80, 90, 100, 110, 120)]
out["fp"]["fig"]["ticks"] = [{"x": float(v), "label": f"{v}"} for v in (50, 100, 150, 200, 250)]

## 6. Exercises

Try each one before looking anything up. Full solutions are in the companion volume, *Part 5: Solutions*.

**Exercise 1.** Show that $E\big[(Y - g(X))^2\big]$ is smallest, over all functions $g$, when $g(X) = E[Y \mid X]$. (Write $Y - g = (Y - E[Y \mid X]) + (E[Y \mid X] - g)$, expand, and use the tower property on the cross term.)

**Exercise 2.** In Longstaff-Schwartz (Lesson 30) the regression uses only the paths that are in the money at that date. Explain why. What goes wrong with the fitted continuation value, where it matters, if the out-of-the-money paths are included?

**Exercise 3.** Three paths have $(X, Y) = (90, 10)$, $(100, 0)$ and $(110, 2)$. Compute the Nadaraya-Watson estimate at $x = 100$ with a normal kernel of bandwidth $h = 10$, and its limits as $h \to 0$ and $h \to \infty$. What does that say about choosing $h$?

**Exercise 4.** Show that the normal density with mean $x_0 + at$ and variance $b^2t$ solves the forward equation $p_t = -a\,p_x + \tfrac12b^2p_{xx}$.

**Exercise 5.** Using the forward equation and integration by parts, show that $\int p(t, x)\,dx$ doesn't change with $t$. Then show that if $g$ solves the backward equation $g_t + a\,g_x + \tfrac12b^2g_{xx} = 0$, the quantity $\int g(t, x)\,p(t, x)\,dx$ doesn't change either. Why does that make forward and backward pricing agree?

**Exercise 6.** Change variables to $x = \ln S_1$, $y = \ln S_2$ in the two-asset equation $V_t + \tfrac12\sigma_1^2S_1^2V_{11} + \tfrac12\sigma_2^2S_2^2V_{22} + \rho\sigma_1\sigma_2S_1S_2V_{12} + rS_1V_1 + rS_2V_2 = rV$, and derive the equation of this lesson. Then compare the work per time step of a full implicit solve (a banded system of $M^2$ unknowns with bandwidth $M$) and of ADI.

**Exercise 7.** (Code) In Longstaff-Schwartz for the American put, replace the polynomial regression with a kernel regression, and compare the prices with Lesson 30's at 50 exercise dates. How sensitive is the answer to the bandwidth?

**Exercise 8.** (Code) Use the Craig-Sneyd scheme for a call on the average of the two shares, $\max\big(\tfrac12(S_1 + S_2) - 100, 0\big)$, with the lesson's parameters, and check the price by Monte Carlo with correlated draws (Lesson 31).

## 7. Export

In [ ]:
path = Path(output_json)
path.parent.mkdir(parents=True, exist_ok=True)
path.write_text(json.dumps(out, indent=2, default=float))
print("wrote", path.resolve(), f"({path.stat().st_size / 1024:.0f} KB)")